# RISHI-Q Kaggle Annotation (GPU)

Blinded structural ontology annotation. Prefer NA over YES. Unity ≠ entanglement.

**Inputs:** attach `rishiq_kaggle_bundle_public` dataset.

**Outputs:** `/kaggle/working/annotations.parquet`, `manifest.json`


In [ ]:
import os, sys, json, platform, time
from pathlib import Path
print(platform.python_version())
try:
    import torch
    print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
except Exception as e:
    print('torch', e)

# Locate bundle
CANDIDATES = [
    Path('/kaggle/input/rishiq-kaggle-bundle-public'),
    Path('/kaggle/input/rishiq_kaggle_bundle_public'),
    Path('../kaggle/bundle'),
    Path('../kaggle/rishiq_kaggle_bundle_public'),
]
INPUT = next((p for p in CANDIDATES if p.exists()), None)
assert INPUT is not None, f'bundle not found in {CANDIDATES}'
OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('../results/exploratory/kaggle_annotation')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('INPUT', INPUT)
print('OUTPUT', OUTPUT)

# Install deps if needed
import subprocess
pkgs = ['pyyaml','pandas','pyarrow','pydantic','numpy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])


In [ ]:
import sys
sys.path.insert(0, str(INPUT / 'rishiq_src'))
import pandas as pd
import yaml
from datetime import datetime, timezone

blinded = pd.read_parquet(INPUT / 'blinded_passages.parquet')
ont = yaml.safe_load((INPUT / 'ontology_v0.1.yaml').read_text())
features = ont['features']
print('passages', len(blinded), 'features', len(features))

# Subsample option for smoke tests
MAX_PASSAGES = int(os.environ.get('RISHIQ_MAX_PASSAGES', '0')) or len(blinded)
# Default: annotate ALL; for smoke set env RISHIQ_MAX_PASSAGES=5
blinded = blinded.head(MAX_PASSAGES).reset_index(drop=True)
print('annotating', len(blinded))


In [ ]:
# Backend selection: transformers if GPU else heuristic fallback
USE_LLM = False
try:
    import torch
    USE_LLM = torch.cuda.is_available()
except Exception:
    USE_LLM = False

if USE_LLM:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'sentencepiece'])
    from rishiq.annotation.transformers_backend import TransformersAnnotationBackend
    from rishiq.models.ontology import Ontology
    from rishiq.models import BlindedPassage
    backend = TransformersAnnotationBackend(model_name=os.environ.get('RISHIQ_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct'))
    ontology = Ontology.model_validate(ont)
    print('backend', backend.name, backend.model_name)
else:
    from rishiq.annotation import HeuristicAnnotationBackend
    from rishiq.models.ontology import Ontology
    from rishiq.models import BlindedPassage
    backend = HeuristicAnnotationBackend()
    ontology = Ontology.model_validate(ont)
    print('backend heuristic (no GPU) — expect floor on literary PD text')


In [ ]:
from rishiq.validation import verify_evidence
rows = []
t0 = time.time()
for i, r in blinded.iterrows():
    bp = BlindedPassage(anonymous_id=r['anonymous_id'], text=r['text'], source_language=r.get('source_language','en'), word_count=int(r.get('word_count') or 0))
    props = backend.extract_propositions(bp)
    anns = backend.annotate_features(bp, props, ontology)
    anns = backend.verify(anns, bp, ontology)
    for a in anns:
        rows.append(a.model_dump(mode='json'))
    if (i+1) % 10 == 0:
        print(f'{i+1}/{len(blinded)} elapsed={time.time()-t0:.1f}s positives={sum(1 for x in rows if x["label"]=="1")}')

ann_df = pd.DataFrame(rows)
ann_path = OUTPUT / 'annotations.parquet'
ann_df.to_parquet(ann_path, index=False)
manifest = {
    'experiment_id': 'kaggle-annotation-pd-pilot',
    'backend': getattr(backend, 'name', 'unknown'),
    'model_name': getattr(backend, 'model_name', getattr(backend, 'name', 'heuristic')),
    'model_revision': getattr(backend, 'revision', 'unspecified'),
    'prompt_version': getattr(backend, 'prompt_version', 'ann-v0.1'),
    'n_passages': len(blinded),
    'n_annotations': len(ann_df),
    'n_positive': int((ann_df['label']=='1').sum()) if len(ann_df) else 0,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'gpu': bool(USE_LLM),
}
(OUTPUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(manifest)
print('wrote', ann_path)
